# EDA on Retail Sales Data
**Oasis Infobyte SIP · Data Analytics · Level 1 · Task 1**

**Objective:** explore a retail sales dataset to uncover sales trends, customer behaviour and product performance, and finish with actionable business recommendations.

**Dataset:** `retail_sales_dataset.csv` (1,000 transactions, Jan-Dec 2023). Columns: Transaction ID, Date, Customer ID, Gender, Age, Product Category, Product Name, Quantity, Price per Unit, Total Amount.

> The file included here is generated by `generate_data.py` (same schema as the popular Kaggle *Retail Sales Dataset*). To use a real Kaggle file instead, drop it in as `retail_sales_dataset.csv`; the code below adapts if `Product Name` is missing.

---
## 1. Setup and data loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("images", exist_ok=True)

df = pd.read_csv("retail_sales_dataset.csv", parse_dates=["Date"])
df.head()

## 2. Initial inspection: shape, dtypes, null values

In [ ]:
print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)
print("\nNull values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate Transaction IDs:", df["Transaction ID"].duplicated().sum())

**Observation:** the dataset has 1,000 rows and 10 columns, with correct dtypes (`Date` parsed as datetime, numeric columns as integers). There are no missing values and no duplicate rows or transaction IDs, so no cleaning is needed before analysis.

## 3. Descriptive statistics (mean, median, mode, standard deviation)

In [ ]:
num_cols = ["Age", "Quantity", "Price per Unit", "Total Amount"]
stats = pd.DataFrame({
    "mean": df[num_cols].mean(),
    "median": df[num_cols].median(),
    "mode": df[num_cols].mode().iloc[0],
    "std": df[num_cols].std(),
    "min": df[num_cols].min(),
    "max": df[num_cols].max(),
}).round(2)
stats

**Observation:** the average transaction is worth about 194, but the median is 120 and the most common value is 80. Mean above median means the distribution is right-skewed: many small baskets plus a few high-value purchases (maximum 1,000, e.g. several units of a high-priced item). The standard deviation (~208) is larger than the mean, showing large variability in basket size. Customer age is centred around the early 40s.

## 4. Time series analysis: monthly and quarterly sales

In [ ]:
ts = df.set_index("Date")["Total Amount"]
monthly = ts.resample("ME").sum()
quarterly = ts.resample("QE").sum()

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(monthly.index.strftime("%b"), monthly.values, marker="o")
ax[0].set_title("Monthly Sales Revenue (2023)")
ax[0].set_ylabel("Total Sales")
ax[1].bar(["Q1", "Q2", "Q3", "Q4"], quarterly.values, color=sns.color_palette("deep")[:4])
ax[1].set_title("Quarterly Sales Revenue (2023)")
plt.tight_layout()
plt.savefig("images/01_sales_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print(monthly.rename("Monthly sales").to_frame().T.to_string())

**Observation:** sales are seasonal. Revenue peaks in **December (~23.4k)**, with **November (~21.0k)** also strong, and there is a spike in **March (~22.8k)**. The weakest months are **February (~10.9k)** and **June (~11.0k)**. Q4 is the best quarter (~60k) and Q2 the weakest (~40k), so Q4 delivers roughly 31% of annual revenue.

## 5. Customer demographics: age groups and gender

In [ ]:
df["Age Group"] = pd.cut(df["Age"], bins=[17, 25, 35, 45, 55, 65],
                         labels=["18-25", "26-35", "36-45", "46-55", "56-64"])

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
sns.countplot(data=df, x="Age Group", ax=ax[0], color="steelblue")
ax[0].set_title("Customers by Age Group")
df["Gender"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax[1], startangle=90)
ax[1].set_title("Gender Split"); ax[1].set_ylabel("")
sns.barplot(data=df, x="Age Group", y="Total Amount", hue="Gender", errorbar=None, ax=ax[2])
ax[2].set_title("Average Spend per Transaction by Age Group and Gender")
plt.tight_layout()
plt.savefig("images/02_demographics.png", dpi=150, bbox_inches="tight")
plt.show()

print(df["Age Group"].value_counts().sort_index())
print(df.groupby("Gender")["Total Amount"].agg(["count", "sum", "mean"]).round(1))

**Observation:** the **36-45** group is the largest (312 transactions, ~31%), followed by 26-35 (237) and 46-55 (227); the youngest (18-25) and oldest (56-64) groups are smallest (~11% each). Gender is balanced (51% female / 49% male). Females spend slightly more in total (~100.6k vs ~92.9k), mostly because of a slightly higher transaction count and average. Interestingly, the **56-64** group has the highest average basket (~230) for both genders, and the **18-25 female** segment spends noticeably more per transaction (~213) than 18-25 males (~153).

## 6. Product analysis: top 10 products and revenue by category

In [ ]:
has_product = "Product Name" in df.columns
if has_product:
    top10 = (df.groupby("Product Name")["Quantity"].sum()
               .sort_values(ascending=False).head(10))
    title = "Top 10 Best-Selling Products (units sold)"
else:  # fall back to category if the dataset has no product names
    top10 = df.groupby("Product Category")["Quantity"].sum().sort_values(ascending=False).head(10)
    title = "Best-Selling Categories (units sold)"

cat_rev = df.groupby("Product Category")["Total Amount"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
top10.sort_values().plot.barh(ax=ax[0], color="teal")
ax[0].set_title(title); ax[0].set_xlabel("Units sold")
cat_rev.plot.bar(ax=ax[1], color=sns.color_palette("deep")[:len(cat_rev)], rot=0)
ax[1].set_title("Revenue by Product Category"); ax[1].set_ylabel("Revenue")
for i, v in enumerate(cat_rev.values):
    ax[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig("images/03_products.png", dpi=150, bbox_inches="tight")
plt.show()

if has_product:
    print(df.groupby("Product Name")["Total Amount"].sum().sort_values(ascending=False).head(5))
print((cat_rev / cat_rev.sum() * 100).round(1))

**Observation:** by **units**, Jackets (253), Phone Cases (242) and Smartwatches (229) lead. By **revenue**, the picture changes: Smartwatches alone bring in ~57k, far above Jackets (~38k) and Perfume (~21k), because of their high unit price. **Electronics** is the top category (~83k, 43% of revenue), followed by **Clothing** (~76k, 39%) and **Beauty** (~35k, 18%). High volume does not always mean high revenue: Phone Cases sell well but add little revenue.

## 7. Correlation heatmap

In [ ]:
corr = df[["Age", "Quantity", "Price per Unit", "Total Amount"]].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Matrix of Numerical Variables")
plt.savefig("images/04_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** `Total Amount` is strongly correlated with `Price per Unit` (0.83) and moderately with `Quantity` (0.41), which is expected since it is their product. `Age` has almost no linear relationship with spending (0.05) or quantity (0.07), and price and quantity are independent (-0.01). Spending is driven by *what* people buy, not *who* they are.

## 8. Additional visualisation: category x gender and day-of-week patterns

In [ ]:
df["Weekday"] = df["Date"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=df, x="Product Category", y="Total Amount", hue="Gender",
            estimator="sum", errorbar=None, ax=ax[0])
ax[0].set_title("Revenue by Category and Gender")

pivot = df.pivot_table(index="Product Category", columns=df["Date"].dt.month,
                       values="Total Amount", aggfunc="sum")
sns.heatmap(pivot, cmap="YlGnBu", ax=ax[1], annot=True, fmt=".0f", cbar=False)
ax[1].set_title("Revenue by Category and Month"); ax[1].set_xlabel("Month")
plt.tight_layout()
plt.savefig("images/05_extra_insights.png", dpi=150, bbox_inches="tight")
plt.show()

print(df.groupby("Weekday")["Total Amount"].mean().reindex(order).round(1))

**Observation:** the category-by-month heatmap shows *where* the seasonal peaks come from, so promotions can be matched to the right category. Females spend more on Clothing and Electronics, while Beauty revenue is essentially equal between genders (~17k each), which challenges the common assumption that Beauty is a female-only segment. Average basket size varies only slightly by weekday (about 181-208), with Mondays and Thursdays highest, so day of week is a weak lever compared with month and product.

## 9. Conclusion and business recommendations

**Key findings**
- Sales are seasonal: Q4 (Nov-Dec) is the strongest period; February and June are weakest.
- Electronics and Clothing generate ~82% of revenue; Smartwatches are the single biggest revenue driver.
- The 36-45 age group is the largest customer base, while 56-64 customers have the highest average basket.
- Age has almost no effect on spend; product price and quantity drive revenue.

**Recommendations**
1. **Plan inventory and marketing around seasonality.** Increase stock and ad spend for Electronics and Clothing ahead of Nov-Dec, and run promotions (bundles, flash sales) in the weak months of February and June to smooth revenue.
2. **Protect and grow high-revenue products.** Keep Smartwatches and Jackets always in stock and feature them prominently. Use low-price, high-volume items such as Phone Cases as add-ons/cross-sells to raise basket value rather than as primary products.
3. **Target the right customer segments.** Focus acquisition on the 36-45 group (largest) and build premium/loyalty offers for 56-64 customers (highest basket). Run campaigns to lift the under-spending 18-25 male segment (e.g. student pricing, bundles).
4. **Use quantity-based incentives.** Since quantity is a major lever on revenue, offer "buy 2, save more" deals on mid-priced items.